# Find roles with Dynamic Form Fields privileges scoped to work items

A Role Editor bug merged Dynamic Form Fields (DFF) management privileges into the Work Items statement when Work Items
was saved before DFF, e.g. `dynamicformfields:Create` on `workitem:workitem` instead of on `*`. These roles cannot
create or update DFF configurations. After the Role Editor fix (helium-uicomponents 1.559.0), DFF shows unchecked for
these roles, and an admin must **select the DFF privileges again and save the role**.

**Detection rule:** a custom role (user or service) is affected if a statement has a resource `workitem:workitem`,
`workorder:workorder` or `workorder:testplan` **and** one of the actions `dynamicformfields:*`, `Create`, `Update`,
`Delete` or `List`. `dynamicformfields:Query` alone is ignored, because Work Items always grants it on work items.

**How to run:** upload this notebook to SystemLink Jupyter and run all cells. It uses the session of the logged-in user
(`SYSTEMLINK_HTTP_URI` / `SYSTEMLINK_API_KEY`), so the user needs permission to read roles.

The notebook only reads roles (`GET /niauth/v1/policy-templates`); it does not modify anything.

In [ ]:
import os
from typing import List
from urllib.parse import urljoin

import pandas as pd
import requests
from requests.adapters import HTTPAdapter, Retry

In [ ]:
# The SystemLink URL and API key of the logged-in user, set by SystemLink Jupyter.
# To run outside SystemLink Jupyter, set these environment variables to an API host
# (e.g. https://<env>-api.systemlink.ni.dev) and an API key that can read roles.
base_url = os.getenv("SYSTEMLINK_HTTP_URI", "")
api_key = os.getenv("SYSTEMLINK_API_KEY", "")


if not base_url or not api_key:
    raise RuntimeError("SYSTEMLINK_HTTP_URI and SYSTEMLINK_API_KEY must be set.")

In [ ]:
class ClientBase:
    def __init__(self):
        retries = Retry(total=5,
                        backoff_factor=2,
                        allowed_methods=None,
                        status_forcelist=[429, 500, 502, 503, 504])
        self._session = requests.Session()
        self._session.mount("http://", HTTPAdapter(max_retries=retries))
        self._session.mount("https://", HTTPAdapter(max_retries=retries))
        # The default requests user agent is blocked by Cloudflare on hosted environments.
        self._session.headers["User-Agent"] = "systemlink-dff-role-audit/1.0"


class AuthServiceClient(ClientBase):
    PAGE_SIZE = 100  # Maximum page size supported by the API

    def __init__(self, base_url: str):
        super().__init__()
        self._base_url = base_url.rstrip("/") + "/"

    def get_policy_templates(self, api_key: str) -> List[dict]:
        """Return all policy templates (user and service roles) of the organization."""
        url = urljoin(self._base_url, "niauth/v1/policy-templates")
        templates = []
        skip = 0
        while True:
            params = {"skip": skip, "take": self.PAGE_SIZE, "sortby": "name", "order": "ascending"}
            r = self._session.get(url, params=params, headers={"x-ni-api-key": api_key})
            r.raise_for_status()
            page = r.json()

            page_templates = page.get("policyTemplates", [])
            templates.extend(page_templates)
            skip += len(page_templates)
            if not page_templates or skip >= page.get("totalCount", 0):
                return templates

In [ ]:
WORK_ITEM_RESOURCES = {"workitem:workitem", "workorder:workorder", "workorder:testplan"}
DFF_MANAGE_ACTIONS = {
    "dynamicformfields:*",
    "dynamicformfields:Create",
    "dynamicformfields:Update",
    "dynamicformfields:Delete",
    "dynamicformfields:List",
}
DFF_CREATE_UPDATE_ACTIONS = {"dynamicformfields:*", "dynamicformfields:Create", "dynamicformfields:Update"}


def find_affected_roles(templates: List[dict]) -> List[dict]:
    """Return custom roles with DFF management actions granted on work item resources instead of "*"."""
    affected = []
    for template in templates:
        if template.get("builtIn"):
            continue
        for statement in template.get("statements") or []:
            resources = set(statement.get("resource") or [])
            dff_actions = set(statement.get("actions") or []) & DFF_MANAGE_ACTIONS
            if dff_actions and resources & WORK_ITEM_RESOURCES:
                affected.append({
                    "Role": template.get("name", ""),
                    "Type": template.get("type", ""),
                    "ID": template.get("id", ""),
                    "Scoped resources": ", ".join(sorted(resources)),
                    "DFF actions": ", ".join(sorted(dff_actions)),
                    "Create/Update blocked": "Yes" if dff_actions & DFF_CREATE_UPDATE_ACTIONS else "No",
                })
    return affected

In [ ]:
auth_client = AuthServiceClient(base_url)


def count_by_type(roles, type_key: str) -> str:
    """Return e.g. "user: 3, service: 1" so both role types are always reported."""
    counts = {"user": 0, "service": 0}
    for role in roles:
        counts[role.get(type_key, "")] = counts.get(role.get(type_key, ""), 0) + 1
    return ", ".join(f"{role_type}: {count}" for role_type, count in counts.items())


def main(api_key: str) -> List[dict]:
    templates = auth_client.get_policy_templates(api_key)
    affected = find_affected_roles(templates)

    custom = [t for t in templates if not t.get("builtIn")]
    print(f"Checked {len(templates)} roles ({len(custom)} custom, {len(templates) - len(custom)} built-in). "
          f"Custom roles by type: {count_by_type(custom, 'type')}.")
    if affected:
        affected_roles = {row["ID"]: row for row in affected}.values()
        print(f"Found {len(affected_roles)} affected role(s) by type: {count_by_type(affected_roles, 'Type')}. "
              "For each role: open it in Security > Roles, select the Dynamic Form Fields privileges again and save.")
    else:
        print("No affected roles found.")
    return affected


affected = main(api_key)


In [ ]:
pd.DataFrame(affected, columns=["Role", "Type", "ID", "Scoped resources", "DFF actions", "Create/Update blocked"])